In [1]:
import pandas as pd
import json
import random
import re
import csv
from pathlib import Path

random.seed(18)

# ----------------------------------
# 1. Templates
# ----------------------------------
AFFIRMED_TEMPLATES = [
    "Patient reports {SYMPTOM}.",
    "Patient complains of {SYMPTOM}.",
    "Patient presents with {SYMPTOM}.",
    "Patient is experiencing {SYMPTOM}.",
    "Patient notes {SYMPTOM}.",
    "Reports {SYMPTOM}.",
    "{SYMPTOM} is present.",
    "Symptoms include {SYMPTOM}.",
    "The patient has {SYMPTOM}.",
    "The patient describes having {SYMPTOM}.",
]

NEGATED_TEMPLATES = [
    "Patient denies {SYMPTOM}.",
    "Patient has no {SYMPTOM}.",
    "No signs of {SYMPTOM}.",
    "No evidence of {SYMPTOM}.",
    "Without any {SYMPTOM}.",
    "The patient does not have {SYMPTOM}.",
    "The patient is free of {SYMPTOM}.",
    "Denies experiencing {SYMPTOM}.",
    "There are no symptoms of {SYMPTOM}.",
    "Not experiencing {SYMPTOM}."
]

# ----------------------------------
# 2. Load symptom dictionary
# ----------------------------------

df = pd.read_csv("base_symptom_dict.csv")  
df.head(2)

,path,level,prefLabel,synonym,definition,word_count_in_prefLabel,id
0,symptom,0,symptom,[],"A symptom is a perceived change in function, s...",1,s0001
1,symptom/musculoskeletal system symptom,1,musculoskeletal system symptom,[],NaN,3,s0002


In [2]:
df[df["id"] == "s0712"]

,path,level,prefLabel,synonym,definition,word_count_in_prefLabel,id
711,symptom/head and neck symptom/head symptom/mou...,4,dry mouth,[],NaN,2,s0712


# Generate Synthetic Samples

In [3]:

# ----------------------------------
# 3. Generate synthetic samples
# ----------------------------------

samples = []

for _, row in df.iterrows():
    symptom_id = row["id"]
    symptom_text = row["prefLabel"]

    # POS examples
    for tmpl in AFFIRMED_TEMPLATES:
        text = tmpl.format(SYMPTOM=symptom_text)
        samples.append({
            "text": text,
            "symptom_id": symptom_id,
            "is_negated": False
        })

    # NEG examples
    for tmpl in NEGATED_TEMPLATES:
        text = tmpl.format(SYMPTOM=symptom_text)
        samples.append({
            "text": text,
            "symptom_id": symptom_id,
            "is_negated": True
        })

# ----------------------------------
# 4. Shuffle
# ----------------------------------

random.shuffle(samples)


# ----------------------------------
# 5. Save to JSONL
# ----------------------------------

def save_jsonl(filename, data):
    with open(filename, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

save_jsonl("synthetic_data.jsonl", samples)

print("Saved:", len(samples))


Saved: 17860


# White Space Tokenization

In [4]:
import pandas as pd
import json
from tqdm import tqdm
# Load the saved datasets back in
def load_jsonl(filename):
    with open(filename, "r") as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl("synthetic_data.jsonl")



In [ ]:
# TOKEN_PATTERN explanation:
# - [A-Za-z0-9]+          : Match a sequence of one or more alphanumeric characters (a word)
# - (?:['-][A-Za-z0-9]+)* : Match zero or more groups where an apostrophe or hyphen is followed by more alphanumerics;
#                           This allows "can't", "mother-in-law", etc. to be tokenized as single words.
# - |                     : OR
# - [^\sA-Za-z0-9]        : Match any single character that is NOT whitespace and is NOT alphanumeric.
#                           This picks up standalone punctuation marks as their own tokens (e.g., ".", ",", "(", ")").
# The overall result: words, possibly including internal apostrophes/hyphens, are single tokens;
# all other non-alphanumeric non-whitespace characters are split as their own tokens.

TOKEN_PATTERN = re.compile(
    r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*|[^\sA-Za-z0-9]"
)

# ----------------------------
# Helpers
# ----------------------------
# Tokenizer that separates words and punctuation into separate tokens.
# Pattern: words with internal apostrophes or hyphens, or any single non-space punctuation char
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*|[^\sA-Za-z0-9]")

def tokenize_with_spans(text):
    """
    Return list of (token, start_char, end_char) using TOKEN_PATTERN.
    Example: "stridor." -> [("stridor", idx, idx+7), (".", idx+7, idx+8)]
    """
    tokens = []
    for m in TOKEN_PATTERN.finditer(text):
        tok = m.group(0)
        tokens.append((tok, m.start(), m.end()))
    return tokens

def normalize_token(tok):
    """Lowercase normalization for matching (leave punctuation tokens as-is)."""
    return tok.lower()

def find_subsequence(token_norms, target_tokens):
    """
    Find first index i where token_norms[i:i+len(target_tokens)] == target_tokens.
    Returns index or None.
    """
    n = len(target_tokens)
    if n == 0:
        return None
    for i in range(len(token_norms) - n + 1):
        ok = True
        for j in range(n):
            if token_norms[i + j] != target_tokens[j]:
                ok = False
                # Don't check the remaining tokens, if first one does not match, we cant match the remaining ones
                break
        if ok:
            return i
    return None

def symptom_to_tokenlist(symptom_text):
    """Convert symptom prefLabel to normalized token list (split on whitespace)."""
    # keep internal hyphens/apostrophes as part of tokens
    parts = [p for p in re.split(r"\s+", symptom_text.strip()) if p]
    parts_norm = [p.lower() for p in parts]
    return parts_norm

# ----------------------------
# Load symptom dictionary (id -> prefLabel)
# ----------------------------
symptom_map = {}
with open("base_symptom_dict.csv", newline='', encoding='utf-8') as f:
    # assume CSV has header and column 'id' and 'prefLabel'
    reader = csv.DictReader(f)
    for r in reader:
        sid = r.get("id") or r.get("symptom_id") or r.get("ID")
        pref = r.get("prefLabel") or r.get("pref_label") or r.get("preflabel")
        if sid is None or pref is None:
            continue
        symptom_map[sid] = pref


In [ ]:

# ----------------------------
# Process input JSONL
# ----------------------------
out_f = open("synthetic_data_tokenized.jsonl", "w", encoding="utf-8")
not_found = []

with open("synthetic_data.jsonl", "r", encoding="utf-8") as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        text = obj["text"]
        sid = obj["symptom_id"]
        is_neg = bool(obj.get("is_negated", False))

        # get symptom text from dict
        symptom_text = symptom_map.get(sid)
        if symptom_text is None:
            # unknown id - skip or record as O-only
            not_found.append({"line": line, "reason": "unknown_symptom_id"})
            print(f"WARNING: NO text was found for {sid}")
            continue

        # 1) Tokenize text into tokens with spans
        tokens_with_spans = tokenize_with_spans(text)
        tokens = [t for (t, s, e) in tokens_with_spans]
        token_norms = [normalize_token(t) for t in tokens]

        # 2) Build symptom token list (normalized)
        symptom_tokens = symptom_to_tokenlist(symptom_text)

        # 3) Try to find symptom as a subsequence in token_norms
        start_idx = find_subsequence(token_norms, symptom_tokens)

        # 4) Fallback: try matching by removing punctuation from token_norms ends (rare)
        if start_idx is None:
            # create versions with punctuation stripped from token ends
            stripped = [re.sub(r'^\W+|\W+$', '', t).lower() for t in tokens]
            start_idx = find_subsequence(stripped, symptom_tokens)

        # 6) If still not found, record and label entire example as O
        labels = ["O"] * len(tokens)
        if start_idx is not None:
            # generate labels B / I
            n = len(symptom_tokens)
            # Map POS/NEG suffix
            suffix = "NEG" if is_neg else "POS"
            b_label = f"B-SYMPTOM_{sid}_{suffix}"
            i_label = f"I-SYMPTOM_{sid}_{suffix}"
            for k in range(n):
                pos = start_idx + k
                if pos < 0 or pos >= len(labels):
                    # safety: skip if out of range
                    continue
                labels[pos] = b_label if k == 0 else i_label
        else:
            not_found.append({"text": text, "symptom_id": sid, "symptom_text": symptom_text})

        # write out tokenized example
        out_obj = {
            "text": text,
            "tokens": tokens,
            "labels": labels,
            "symptom_id": sid,
            "is_negated": is_neg
        }
        out_f.write(json.dumps(out_obj, ensure_ascii=False) + "\n")

out_f.close()

# ----------------------------
# Report
# ----------------------------

print("Total not found / ambiguous matches:", len(not_found))
if len(not_found) > 0:
    # show a few
    print("Examples of not-found:")
    for e in not_found[:10]:
        print(e)

Total not found / ambiguous matches: 0


# Learning

In [8]:

# Simple tests for the TOKEN_PATTERN
test_cases = [
    "Hello, world!",
    "It's John's well-known brother-in-law.",
    "3.141-5, 'quoted', hyphen-word",
    "Can't—won't... isn't.",
    "Email me at test-email@example.com!",
    "-punctuation- 'standalone' --double-hyphens--",
]

# for text in test_cases:
#     tokens = [m.group(0) for m in TOKEN_PATTERN.finditer(text)]
#     print(f"Input: {text}")
#     print(f"Tokens: {tokens}")
#     print("----")

tokens_with_spans = tokenize_with_spans("My body is a system, a musculoskeletal system symptom")
tokens_with_spans

[('My', 0, 2),
 ('body', 3, 7),
 ('is', 8, 10),
 ('a', 11, 12),
 ('system', 13, 19),
 (',', 19, 20),
 ('a', 21, 22),
 ('musculoskeletal', 23, 38),
 ('system', 39, 45),
 ('symptom', 46, 53)]

In [9]:
token_norms = [normalize_token(t) for (t, s, e) in tokens_with_spans]
token_norms

['my',
 'body',
 'is',
 'a',
 'system',
 ',',
 'a',
 'musculoskeletal',
 'system',
 'symptom']

In [10]:
def symptom_to_tokenlist(symptom_text):
    """Convert symptom prefLabel to normalized token list (split on whitespace)."""
    # keep internal hyphens/apostrophes as part of tokens
    parts = [p for p in re.split(r"\s+", symptom_text.strip()) if p]
    parts_norm = [p.lower() for p in parts]
    return parts_norm
    
tok_symp = symptom_to_tokenlist("musculoskeletal system symptom.")
tok_symp

['musculoskeletal', 'system', 'symptom.']

In [ ]:
def find_subsequence(token_norms, target_tokens):
    """
    Find the first index i in token_norms where a slice of length len(target_tokens) matches the target_tokens exactly.
    Returns the starting index, or None if not found.
    
    This uses a sliding window approach: we check each possible starting position to see if
    the next n tokens match our target sequence.
    """

    # Number of tokens that we are searching for
    n = len(target_tokens)
    if n == 0:
        return None

    # Slide a window of size n across token_norms
    # Example: if token_norms has 10 tokens and n=3, we check positions 0-7 (8 possible windows)
    for i in range(len(token_norms) - n + 1):
        ok = True  # Optimistically assume this window will match until proven otherwise
        
        # Check each token in the current window [i, i+1, ..., i+n-1]
        for j in range(n):
            # Compare token at position (i+j) in token_norms with token at position j in target_tokens
            print(f"i,j = {i},{j}")
            print("token_norms[i + j]: ", token_norms[i + j])
            print("target_tokens[j]", target_tokens[j])
            
            if token_norms[i + j] != target_tokens[j]:
                # Mismatch found! This window cannot be a match.
                # We break early because there's no point checking the remaining tokens
                # in this window - we already know it won't match.
                ok = False
                print("break loop")
                break  # Exit inner loop immediately, move to next window position (i+1)
        
        # If we made it through all n comparisons without breaking, all tokens matched!
        if ok:
            # If all tokens matched, return the start index
            return i
    
    # No match found after checking all possible window positions
    return None

# Example usage: find the start index where symptom_tokens appear in token_norms
print(f"token norms ({len(token_norms)}): {token_norms}\nTarget tokens ({len(tok_symp)}): {tok_symp}")
start_idx = find_subsequence(token_norms, tok_symp)
print(start_idx)

token norms (10): ['my', 'body', 'is', 'a', 'system', ',', 'a', 'musculoskeletal', 'system', 'symptom']
Target tokens (3): ['musculoskeletal', 'system', 'symptom.']
i,j = 0,0
token_norms[i + j]:  my
target_tokens[j] musculoskeletal
break loop
i,j = 1,0
token_norms[i + j]:  body
target_tokens[j] musculoskeletal
break loop
i,j = 2,0
token_norms[i + j]:  is
target_tokens[j] musculoskeletal
break loop
i,j = 3,0
token_norms[i + j]:  a
target_tokens[j] musculoskeletal
break loop
i,j = 4,0
token_norms[i + j]:  system
target_tokens[j] musculoskeletal
break loop
i,j = 5,0
token_norms[i + j]:  ,
target_tokens[j] musculoskeletal
break loop
i,j = 6,0
token_norms[i + j]:  a
target_tokens[j] musculoskeletal
break loop
i,j = 7,0
token_norms[i + j]:  musculoskeletal
target_tokens[j] musculoskeletal
i,j = 7,1
token_norms[i + j]:  system
target_tokens[j] system
i,j = 7,2
token_norms[i + j]:  symptom
target_tokens[j] symptom.
break loop
None


In [11]:
symptom_tokens

['skin', 'desquamation']